# Week 7: Introduction to optimisation
ENGI 1151 Computational Tools for Engineers and Scientists

Run each cell with **Shift + Enter**, from top to bottom: later cells use variables from earlier ones. The full explanations are on the week 7 notes page.

## 3. Closed form, one variable
The stationary points of $f(x) = \cos(2x)\,e^{-0.1x^2}$: where $f'(x) = 0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

f = lambda x: np.cos(2 * x) * np.exp(-0.1 * x**2)
df = lambda x: -np.exp(-0.1 * x**2) * (2 * np.sin(2 * x)
                                     + 0.2 * x * np.cos(2 * x))

x = np.linspace(-5, 5, 1000)
# stationary points: f' changes sign between two grid points,
# then brentq refines each one
i = np.where(df(x[:-1]) * df(x[1:]) < 0)[0]
xs = np.array([brentq(df, x[j], x[j + 1]) for j in i])
print("stationary points: x =", np.round(xs, 3))
print("f at these points:    ", np.round(f(xs), 3))
best = xs[np.argmax(f(xs))]
print("global maximum: x = %.3f, f = %.3f" % (best, f(best)))

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(7, 5))
ax[0].plot(x, f(x), color="#68246D")
ax[0].plot(xs, f(xs), "o", color="#B3261E")
ax[0].set_ylabel("f(x)")
ax[1].plot(x, df(x), color="#1D5FA3")
ax[1].axhline(0, color="black", lw=0.8)
ax[1].plot(xs, df(xs), "o", color="#B3261E")
ax[1].set_ylabel("f'(x)")
ax[1].set_xlabel("x")
plt.tight_layout()
plt.show()


## 4. Closed form, two variables
Solve $\nabla h = \mathbf{0}$ for the smooth mountain, a linear system.

In [ ]:
import numpy as np

# -60x + 10y = -210  and  10x - 40y = -80
A = np.array([[-60, 10],
              [10, -40]])
b = np.array([-210, -80])
x_star, y_star = np.linalg.solve(A, b)

h = lambda x, y: -30*x**2 - 20*y**2 + 10*x*y + 210*x + 80*y + 660
print("peak at x = %.2f km, y = %.2f km, height %.0f m"
      % (x_star, y_star, h(x_star, y_star)))


## 6. Gradient descent by hand
Minimise $f(x) = x\cos 3x$ from $x_0 = -3.9$. Change `alpha` and `x` and run again.

In [ ]:
import numpy as np

def f(x):
    return x * np.cos(3 * x)

def grad(x):                    # f'(x)
    return np.cos(3 * x) - 3 * x * np.sin(3 * x)

x = -3.9                        # initial guess
alpha = 0.02                    # learning rate
print(" k         x        f(x)       f'(x)")
for k in range(1, 51):
    step = alpha * grad(x)
    x = x - step                # move against the gradient
    print("%2d %10.5f %10.5f %10.5f" % (k, x, f(x), grad(x)))
    if abs(step) < 1e-6:        # converged: tiny step
        break


Gradient descent paths for different learning rates and starting points.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return x * np.cos(3 * x)

def grad(x):
    return np.cos(3 * x) - 3 * x * np.sin(3 * x)

def gradient_descent(x, alpha, n_max=50, tol=1e-6):
    path = [x]
    for k in range(n_max):
        step = alpha * grad(x)
        x = x - step
        path.append(x)
        if abs(step) < tol:
            break
    return np.array(path)

xs = np.linspace(-5, 2, 500)
plt.plot(xs, f(xs), color="black", lw=1.5)
runs = [(-3.9, 0.02, "#68246D"), (-3.9, 0.07, "#B3261E"), (0.5, 0.02, "#1D5FA3")]
for x0, alpha, colour in runs:
    path = gradient_descent(x0, alpha)
    print("x0 = %5.2f, alpha = %.2f: %2d iterations, end at x = %.4f"
          % (x0, alpha, len(path) - 1, path[-1]))
    plt.plot(path, f(path), "o-", ms=4, color=colour,
             label="x0 = %g, alpha = %g" % (x0, alpha))
plt.xlim(-5, 2)
plt.ylim(-4.6, 4.2)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.show()


## 7. Newton's method by hand
Try `x = 0.5`: where does it converge, and what is the sign of $f''$ there?

In [ ]:
import numpy as np

def f(x):
    return x * np.cos(3 * x)

def grad(x):                    # f'(x)
    return np.cos(3 * x) - 3 * x * np.sin(3 * x)

def hessian(x):                 # f''(x)
    return -6 * np.sin(3 * x) - 9 * x * np.cos(3 * x)

x = -3.9                        # initial guess
print(" k         x        f(x)      f'(x)   f''(x)")
for k in range(1, 21):
    step = grad(x) / hessian(x)
    x = x - step                # stationary point of the parabola
    print("%2d %10.5f %10.5f %10.1e %8.2f"
          % (k, x, f(x), grad(x), hessian(x)))
    if abs(step) < 1e-8:
        break


## 8. `scipy.optimize`
### 8.2 `minimize` with CG and Newton-CG

In [ ]:
import numpy as np
from scipy.optimize import minimize

def f(x):                        # x is an array: x[0] is our variable
    return x[0] * np.cos(3 * x[0])
def jac(x):                      # gradient: an array
    t = x[0]
    return np.array([np.cos(3 * t) - 3 * t * np.sin(3 * t)])
def hess(x):                     # Hessian: a matrix
    t = x[0]
    return np.array([[-6 * np.sin(3 * t) - 9 * t * np.cos(3 * t)]])

x0 = np.array([-3.9])
res_cg = minimize(f, x0, method="CG", jac=jac)
res_nt = minimize(f, x0, method="Newton-CG", jac=jac, hess=hess)

for name, r in [("CG", res_cg), ("Newton-CG", res_nt)]:
    print("%-9s x = %.5f  f = %.5f  nit = %d  success = %s"
          % (name, r.x[0], r.fun, r.nit, r.success))


### 8.4 Bounds and maximisation

In [ ]:
import numpy as np
from scipy.optimize import minimize

f = lambda x: x[0] * np.cos(3 * x[0])

free = minimize(f, x0=[-4], method="BFGS")
bounded = minimize(f, x0=[-4], method="L-BFGS-B", bounds=[(-4, 2)])
print("BFGS, no bounds:       x = %.4f, f = %.4f" % (free.x[0], free.fun))
print("L-BFGS-B, -4 <= x <= 2: x = %.4f, f = %.4f"
      % (bounded.x[0], bounded.fun))


In [ ]:
import numpy as np
from scipy.optimize import minimize

f = lambda x: np.cos(2 * x[0]) * np.exp(-0.1 * x[0]**2)

res = minimize(lambda x: -f(x), x0=[0.4])
print("maximum at x = %.4f, f = %.4f" % (res.x[0], -res.fun))


## 9. Climbing the smooth mountain in 2-D
The mountain, its gradient and its Hessian:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def h(p):                       # height of the smooth mountain (m)
    x, y = p
    return -30*x**2 - 20*y**2 + 10*x*y + 210*x + 80*y + 660

def grad_h(p):                  # gradient (Jacobian) of h
    x, y = p
    return np.array([-60*x + 10*y + 210, 10*x - 40*y + 80])

def hess_h(p):                  # Hessian of h (constant)
    return np.array([[-60.0, 10.0], [10.0, -40.0]])


Gradient ascent:

In [ ]:
x = np.array([0.5, 5.5])        # initial guess (km)
alpha = 0.01                    # learning rate
path = [x]
print(" k       x       y    height   |grad|")
for k in range(1, 101):
    step = alpha * grad_h(x)
    x = x + step                # uphill: gradient ascent
    path.append(x)
    print("%2d %7.3f %7.3f %8.1f %8.2f"
          % (k, x[0], x[1], h(x), np.linalg.norm(grad_h(x))))
    if np.linalg.norm(step) < 1e-4:
        break
path = np.array(path)

X, Y = np.meshgrid(np.linspace(0, 8, 200), np.linspace(0, 6, 200))
plt.contourf(X, Y, h([X, Y]), levels=15, cmap="Purples")
plt.colorbar(label="height h (m)")
plt.plot(path[:, 0], path[:, 1], "o-", color="#B3261E", ms=4,
         label="gradient ascent")
plt.plot(4, 3, "*", color="gold", ms=16, mec="black", label="peak (4, 3)")
plt.xlabel("x (km)")
plt.ylabel("y (km)")
plt.legend()
plt.show()


Newton's method: one step is enough for a quadratic function.

In [ ]:
x = np.array([0.5, 5.5])
for k in range(1, 4):
    step = np.linalg.solve(hess_h(x), grad_h(x))   # H step = gradient
    x = x - step
    print("%d  x = %.4f, y = %.4f, height = %.2f, |grad| = %.1e"
          % (k, x[0], x[1], h(x), np.linalg.norm(grad_h(x))))


The same with SciPy, minimising $-h$:

In [ ]:
from scipy.optimize import minimize

res = minimize(lambda p: -h(p), x0=[0.5, 5.5],
               jac=lambda p: -grad_h(p), method="BFGS")
print("peak at", np.round(res.x, 4), "height %.1f m" % -res.fun)
print("iterations:", res.nit, "success:", res.success)


## 10. The best spacing of two wind turbines
The Jensen wake model and the objective, power minus the cost of spacing:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def deficit(s, Ct=0.8, k=0.05):
    """Jensen wake: wind speed deficit s rotor diameters behind a turbine."""
    return (1 - np.sqrt(1 - Ct)) / (1 + 2*k*s)**2

def objective(s, Ct=0.8, k=0.05, cost=0.04):
    """Power of two turbines (units of one turbine) minus the cost of spacing s."""
    return 1 + (1 - deficit(s, Ct, k))**3 - cost * s


In [ ]:
from scipy.optimize import minimize

res = minimize(lambda s: -objective(s[0]), x0=[3],
               method="L-BFGS-B", bounds=[(2, 15)])
s_best = res.x[0]
print("best spacing: %.2f rotor diameters" % s_best)
print("objective: %.3f, success: %s" % (-res.fun, res.success))
P2 = (1 - deficit(s_best))**3
print("power of the second turbine: %.0f %% of the first" % (100 * P2))

s = np.linspace(2, 15, 300)
plt.plot(s, 1 + (1 - deficit(s))**3, label="power of both turbines")
plt.plot(s, objective(s), lw=2.5, color="#68246D", label="objective = power − cost")
plt.axvline(s_best, color="gray", ls="--")
plt.plot(s_best, -res.fun, "*", ms=16, color="gold", mec="black")
plt.xlabel("spacing s (rotor diameters)")
plt.ylabel("power (units of one turbine)")
plt.legend()
plt.show()


Sensitivity of the best spacing to the cost of spacing:

In [ ]:
from scipy.optimize import minimize

for cost in [0.02, 0.03, 0.04, 0.05, 0.06]:
    res = minimize(lambda s: -objective(s[0], cost=cost), x0=[3],
                   method="L-BFGS-B", bounds=[(2, 15)])
    print("cost %.2f per D: best spacing %5.2f D" % (cost, res.x[0]))


## Exercises
1. Run the gradient descent of section 6 with `alpha` = 0.01, 0.04, 0.07 and 0.12, and from `x0` = 0.5 and −2. Describe what happens in each case.
2. Run both hand-written methods on $f(x) = x^2$ (derivatives $2x$ and $2$). Why does Newton's method need only one step?
3. Maximise $f(x) = \cos(2x)\,e^{-0.1x^2}$ with `minimize` from $x_0 = 2.5$. Which peak do you find? Repeat from 20 starting points between −5 and 5 and keep the best.
4. Run the 2-D gradient ascent on the mountain with `alpha` = 0.02, 0.03 and 0.035. How many iterations does each need? What happens with 0.035, and why?
5. Find the best turbine spacing offshore, where the wake decays more slowly (`k = 0.04`), with `minimize` and with `minimize_scalar`.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3


In [ ]:
# Exercise 4


In [ ]:
# Exercise 5
